# Triagem de Saúde Mental — Somente a parte de Machine Learning

Este notebook contém **apenas a IA** do projeto "Triagem Inteligente para
Otimização da Saúde Mental" (sem Flask, sem site, sem banco de dados, sem
envio de e-mail). Serve para rodar no **Google Colab** ou em qualquer
Python local, treinar os modelos, gerar gráficos e testar se a IA está
aprendendo direito.

**O que tem aqui:**
1. Carregar e explorar os dados
2. Treinar um modelo **Naive Bayes** (pedido) e comparar com o **Random
   Forest** (o que o projeto original usava)
3. Avaliar com matriz de confusão, precisão, recall e F1
4. Gráficos: comparação dos modelos, curva de aprendizado, importância das
   variáveis
5. Uma função para você simular um caso manualmente e ver o que a IA prevê
6. Testes de sanidade automáticos (casos extremos) para saber se a IA está
   "puxando os valores certos"
7. Validação cruzada, para saber se o resultado é estável ou só sorte do
   sorteio treino/teste

**Antes de rodar no Colab:** faça upload do arquivo
`StressLevelDataset.csv` (está junto com este notebook) na primeira
célula de código quando for pedido.

In [ ]:
# Bibliotecas (todas já vêm instaladas por padrão no Google Colab)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import (
    train_test_split, learning_curve, StratifiedKFold, cross_val_score
)
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.inspection import permutation_importance
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Carregar os dados

Dataset usado: `StressLevelDataset.csv` — questionário psicossocial real
de estudantes (1100 respostas, 21 colunas, sem valores nulos). Foi o
dataset que o projeto original escolheu porque é o único com correlação
real entre os fatores e os níveis de ansiedade/depressão relatados.

In [ ]:
CSV_PATH = "StressLevelDataset.csv"

if not os.path.exists(CSV_PATH):
    try:
        from google.colab import files
        print("Arquivo não encontrado na pasta atual. Faça upload do StressLevelDataset.csv:")
        uploaded = files.upload()
        CSV_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            "Coloque o StressLevelDataset.csv na mesma pasta deste notebook."
        )

df = pd.read_csv(CSV_PATH)
print("Formato:", df.shape)
print("Valores nulos:", df.isnull().sum().sum())
df.head()

## 2. Análise exploratória rápida

Antes de treinar qualquer modelo, vale olhar a correlação de cada fator
com o nível de estresse. Isso também vai servir mais na frente, na seção
de testes de sanidade (célula 12).

In [ ]:
FEATURES = [
    "headache", "blood_pressure", "sleep_quality", "breathing_problem",
    "noise_level", "living_conditions", "safety", "basic_needs",
    "academic_performance", "study_load", "teacher_student_relationship",
    "future_career_concerns", "social_support", "peer_pressure",
    "extracurricular_activities", "bullying", "mental_health_history",
]

corr = df[FEATURES + ["stress_level"]].corr()["stress_level"].drop("stress_level").sort_values()

fig, ax = plt.subplots(figsize=(7, 6))
corr.plot(kind="barh", ax=ax, color=["crimson" if v > 0 else "steelblue" for v in corr])
ax.set_title("Correlação de cada fator com o nível de estresse")
ax.set_xlabel("Correlação (azul = protege, vermelho = aumenta o risco)")
plt.tight_layout()
plt.show()

## 3. Criar os rótulos de risco (o que a IA vai aprender a prever)

O dataset traz escores numéricos de ansiedade (0-21, estilo GAD-7) e
depressão (0-27, estilo PHQ-9). Convertemos em faixas clínicas
Baixo/Médio/Alto — a mesma lógica usada em sala de aula para transformar
um valor contínuo em classes (igual a Aula 06, onde a Renda virava
faixas `<15k`, `15k-35k`, `>35k`).

É uma classificação **multirrótulo**: um estudante pode estar com risco
Alto de ansiedade **e** de depressão ao mesmo tempo — são 3 previsões
independentes saindo do mesmo modelo.

In [ ]:
def faixa_ansiedade(score):  # anxiety_level: 0-21
    if score < 8:
        return "Baixo"
    elif score < 15:
        return "Medio"
    return "Alto"


def faixa_depressao(score):  # depression: 0-27
    if score < 10:
        return "Baixo"
    elif score < 20:
        return "Medio"
    return "Alto"


MAPA_ESTRESSE = {0: "Baixo", 1: "Medio", 2: "Alto"}  # stress_level já vem 0/1/2

df["risco_ansiedade"] = df["anxiety_level"].apply(faixa_ansiedade)
df["risco_depressao"] = df["depression"].apply(faixa_depressao)
df["risco_estresse"] = df["stress_level"].map(MAPA_ESTRESSE)

LABEL_COLS = ["risco_ansiedade", "risco_depressao", "risco_estresse"]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, col in zip(axes, LABEL_COLS):
    df[col].value_counts().reindex(["Baixo", "Medio", "Alto"]).plot(kind="bar", ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 4. Separar treino e teste

**Hold-out** 80/20, igual ao visto na Aula 07 (slide 17). O modelo nunca
vê os dados de teste durante o treino — é a única forma honesta de medir
se ele generaliza.

In [ ]:
X = df[FEATURES].copy()

encoders = {}
y = pd.DataFrame()
for col in LABEL_COLS:
    le = LabelEncoder()
    y[col] = le.fit_transform(df[col])
    encoders[col] = le

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y["risco_estresse"]
)
print("Treino:", X_train.shape[0], "| Teste:", X_test.shape[0])

## 5. Treinar o Naive Bayes

Usamos `GaussianNB` (Naive Bayes Gaussiano): como os fatores aqui são
escalas numéricas (0-5, 0-3 etc.), em vez das categorias "Boa/Ruim/
Desconhecida" do exemplo de Risco de Crédito da Aula 05, o Naive Bayes
Gaussiano assume que cada fator segue uma distribuição normal dentro de
cada classe — é a versão do algoritmo adequada para esse tipo de dado,
mas o princípio é o mesmo: calcular a probabilidade de cada classe e
escolher a mais provável (Teorema de Bayes).

`MultiOutputClassifier` treina um Naive Bayes separado para cada um dos
3 rótulos (ansiedade, depressão, estresse), do mesmo jeito que o projeto
original fazia com o Random Forest.

In [ ]:
pipeline_nb = Pipeline(steps=[
    ("preprocessador", ColumnTransformer([("num", StandardScaler(), FEATURES)])),
    ("modelo", MultiOutputClassifier(GaussianNB())),
])
pipeline_nb.fit(X_train, y_train)
y_pred_nb = pipeline_nb.predict(X_test)
print("Naive Bayes treinado.")

## 6. Treinar também o Random Forest (para comparar)

O projeto original usava Random Forest (floresta de Árvores de Decisão —
Aula 06). Treinamos ele também aqui só para servir de comparação lado a
lado com o Naive Bayes — não é obrigatório, mas ajuda a responder "o
Naive Bayes é bom o suficiente?".

In [ ]:
pipeline_rf = Pipeline(steps=[
    ("preprocessador", ColumnTransformer([("num", StandardScaler(), FEATURES)])),
    ("modelo", MultiOutputClassifier(RandomForestClassifier(
        n_estimators=150, max_depth=8, min_samples_leaf=3,
        class_weight="balanced", random_state=RANDOM_STATE
    ))),
])
pipeline_rf.fit(X_train, y_train)
y_pred_rf = pipeline_rf.predict(X_test)
print("Random Forest treinado.")

## 7. Avaliar os modelos — Matriz de Confusão, Precisão, Recall, F1

Isso é exatamente o conteúdo da Aula 07. `classification_report` já
calcula Precisão, Recall e F1 por classe automaticamente.

In [ ]:
for nome, y_pred in [("NAIVE BAYES", y_pred_nb), ("RANDOM FOREST", y_pred_rf)]:
    print("=" * 70)
    print(nome)
    print("=" * 70)
    for i, col in enumerate(LABEL_COLS):
        print(f"\n--- {col} ---")
        print(classification_report(
            y_test[col], y_pred[:, i], target_names=encoders[col].classes_, zero_division=0
        ))

**Como ler isso:** `precision` (precisão) responde "quando a IA disse
Alto, com que frequência ela acertou?". `recall` responde "de todos os
casos que eram realmente Alto, quantos a IA pegou?". `f1-score` é o
equilíbrio dos dois. Quanto mais perto de 1.00, melhor — e preste atenção
se alguma classe específica (ex.: "Alto") está com recall muito mais
baixo que as outras, porque é a classe que mais importa não deixar passar
batido numa triagem de saúde.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for linha, (nome, y_pred) in enumerate([("Naive Bayes", y_pred_nb), ("Random Forest", y_pred_rf)]):
    for i, col in enumerate(LABEL_COLS):
        cm = confusion_matrix(y_test[col], y_pred[:, i])
        ax = axes[linha, i]
        ax.imshow(cm, cmap="Blues")
        labels = encoders[col].classes_
        ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
        ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
        ax.set_xlabel("Previsto"); ax.set_ylabel("Real")
        ax.set_title(f"{nome} — {col}")
        for r in range(cm.shape[0]):
            for c in range(cm.shape[1]):
                ax.text(c, r, cm[r, c], ha="center", va="center",
                        color="white" if cm[r, c] > cm.max() / 2 else "black")
plt.tight_layout()
plt.show()

**Como ler a matriz de confusão:** a diagonal principal (de cima-esquerda
a baixo-direita) é onde a IA acertou. Qualquer número fora da diagonal é
um erro — e quanto mais longe da diagonal (ex.: previu "Baixo" quando era
"Alto"), mais grave é o erro num contexto de triagem de saúde.

## 8. Comparar visualmente Naive Bayes x Random Forest

In [ ]:
f1_nb = [f1_score(y_test[c], y_pred_nb[:, i], average="macro") for i, c in enumerate(LABEL_COLS)]
f1_rf = [f1_score(y_test[c], y_pred_rf[:, i], average="macro") for i, c in enumerate(LABEL_COLS)]

x = np.arange(len(LABEL_COLS))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width/2, f1_nb, width, label="Naive Bayes")
ax.bar(x + width/2, f1_rf, width, label="Random Forest")
ax.set_xticks(x); ax.set_xticklabels(LABEL_COLS)
ax.set_ylabel("F1-score (macro)")
ax.set_ylim(0, 1)
ax.set_title("Naive Bayes x Random Forest")
ax.legend()
plt.tight_layout()
plt.show()

print("F1 macro Naive Bayes:", [round(v, 3) for v in f1_nb])
print("F1 macro Random Forest:", [round(v, 3) for v in f1_rf])

## 9. A IA está realmente aprendendo? (Curva de Aprendizado)

Essa é a pergunta mais importante. A **curva de aprendizado** treina o
modelo várias vezes, com quantidades crescentes de dados, e mede o
desempenho no treino e numa validação separada a cada passo (conceito
parecido com o gráfico de overfitting/underfitting da Aula 06, slide 23).

**Como interpretar:**
- Se a linha de **Validação** sobe conforme mais dados são usados e se
  aproxima da linha de **Treino** → a IA está aprendendo padrões reais.
- Se a linha de Treino está muito mais alta que a de Validação, com uma
  distância grande entre elas → **overfitting** (decorou o treino, não
  generaliza).
- Se as duas linhas ficam baixas e próximas → **underfitting** (o modelo
  é simples demais ou faltam dados/fatores informativos).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, LABEL_COLS):
    modelo_simples = Pipeline([("prep", StandardScaler()), ("nb", GaussianNB())])
    train_sizes, train_scores, val_scores = learning_curve(
        modelo_simples, X, y[col], cv=5, scoring="f1_macro",
        train_sizes=np.linspace(0.1, 1.0, 8), random_state=RANDOM_STATE
    )
    ax.plot(train_sizes, train_scores.mean(axis=1), "o-", label="Treino")
    ax.plot(train_sizes, val_scores.mean(axis=1), "o-", label="Validação")
    ax.set_title(col)
    ax.set_xlabel("Exemplos de treino")
    ax.set_ylabel("F1 macro")
    ax.legend()
plt.tight_layout()
plt.show()

## 10. Validação cruzada (k-fold) — o resultado é estável ou foi sorte?

Igual ao conceito da Aula 07 (slide 18, "Validação cruzada r=3"), mas
aqui com 5 divisões (5-fold). Em vez de testar com uma única divisão
treino/teste, dividimos os dados em 5 partes, treinamos 5 vezes (cada
vez deixando uma parte diferente de fora para teste) e olhamos a **média
e o desvio padrão**. Desvio padrão pequeno = resultado consistente,
independente de qual pedaço dos dados caiu no teste.

In [ ]:
print("VALIDAÇÃO CRUZADA (5-fold) — Naive Bayes")
for col in LABEL_COLS:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    modelo_simples = Pipeline([("prep", StandardScaler()), ("nb", GaussianNB())])
    scores = cross_val_score(modelo_simples, X, y[col], cv=skf, scoring="f1_macro")
    print(f"{col}: média={scores.mean():.3f}  desvio={scores.std():.3f}  | folds: {np.round(scores, 3)}")

## 11. Testar manualmente — simule um caso e veja o que a IA prevê

Essa função recebe só os fatores que você quiser mudar (o resto fica na
**média** do dataset) e devolve a previsão da IA para os 3 riscos. É a
forma mais direta de responder "a IA está puxando os valores certos?":
você simula um caso que você já sabe (pelo bom senso) qual deveria ser o
resultado, e confere se a previsão bate.

As colunas do questionário, todas em escala 0-5 (exceto
`blood_pressure`: 1-3, `social_support`: 0-3, `mental_health_history`:
0 ou 1):

`headache, blood_pressure, sleep_quality, breathing_problem, noise_level,
living_conditions, safety, basic_needs, academic_performance, study_load,
teacher_student_relationship, future_career_concerns, social_support,
peer_pressure, extracurricular_activities, bullying,
mental_health_history`

In [ ]:
MEDIAS = X.mean().to_dict()


def prever_caso(pipeline, **kwargs):
    """Simula um caso: parte da média do dataset e só muda o que você passar.
    Ex: prever_caso(pipeline_nb, sleep_quality=0, bullying=5, safety=0)
    """
    caso = MEDIAS.copy()
    caso.update(kwargs)
    linha = pd.DataFrame([caso])[FEATURES]
    pred = pipeline.predict(linha)[0]
    return {col: encoders[col].inverse_transform([pred[i]])[0] for i, col in enumerate(LABEL_COLS)}


# Exemplo 1: caso médio (ninguém preencheu nada "fora do comum")
print("Caso médio:", prever_caso(pipeline_nb))

# Exemplo 2: só 3 fatores ruins (bullying, sono, segurança), o resto na média
print("3 fatores ruins:", prever_caso(pipeline_nb, bullying=5, sleep_quality=0, safety=0))

# Exemplo 3: 7 fatores bons (suporte, sono, segurança, moradia, notas, professores)
print("7 fatores bons:", prever_caso(
    pipeline_nb, social_support=3, sleep_quality=5, safety=5, basic_needs=5,
    academic_performance=5, teacher_student_relationship=5, living_conditions=5
))

**Resultado real desses dois testes:** com só 3 fatores ruins, o
resultado ainda sai "Médio" nos três riscos — porque o Naive Bayes
multiplica a contribuição de **todos** os 17 fatores, e os outros 14
continuam na média, "diluindo" o efeito dos 3 ruins. Já com 7 fatores
bons o resultado já vira "Baixo" em tudo.

Isso não é um bug: é a desvantagem do Naive Bayes mencionada na Aula 05
(slide 9) — ele trata cada fator como **independente** dos demais e
multiplica as probabilidades, então poucos fatores extremos pesam menos
do que mudar vários fatores ao mesmo tempo. É por isso que a seção 12
usa os **17 fatores no extremo** para montar o teste de sanidade — só
assim dá pra isolar se o modelo consegue, no limite, reconhecer o pior e
o melhor cenário possível.

Mude os números acima (ou copie a célula) e teste os seus próprios
cenários.

## 12. Testes de sanidade automáticos (casos extremos)

Em vez de confiar só no "olhômetro", este teste usa a correlação de cada
fator com o estresse (calculada na seção 2) para montar automaticamente:

- um **pior caso possível** (cada fator no valor que mais aumenta o
  risco, segundo os dados);
- um **melhor caso possível** (cada fator no valor mais protetivo).

Se a IA estiver aprendendo direito, o pior caso **tem** que sair como
"Alto" nos três riscos, e o melhor caso **tem** que sair como "Baixo".
Isso é um teste automático de regressão: toda vez que você re-treinar o
modelo (com mais dados, por exemplo), rode essa célula de novo — se
algum dia parar de passar, é sinal de alerta de que algo quebrou no
treino.

In [ ]:
piora_se_sobe = corr[corr > 0].index.tolist()
melhora_se_sobe = corr[corr < 0].index.tolist()

maximos, minimos = X.max().to_dict(), X.min().to_dict()

pior_caso = {f: maximos[f] for f in piora_se_sobe}
pior_caso.update({f: minimos[f] for f in melhora_se_sobe})

melhor_caso = {f: minimos[f] for f in piora_se_sobe}
melhor_caso.update({f: maximos[f] for f in melhora_se_sobe})


def checar(nome, previsto, esperado):
    ok = all(v == esperado for v in previsto.values())
    status = "PASSOU" if ok else "FALHOU"
    print(f"[{status}] {nome}: {previsto}  (esperado: tudo '{esperado}')")
    return ok


resultado_pior = prever_caso(pipeline_nb, **pior_caso)
resultado_melhor = prever_caso(pipeline_nb, **melhor_caso)

checar("Pior caso simulado", resultado_pior, "Alto")
checar("Melhor caso simulado", resultado_melhor, "Baixo")

## 13. Importância das variáveis — o que mais pesa na decisão

Usa `permutation_importance`: embaralha os valores de uma coluna por vez
e mede o quanto o desempenho do modelo piora. Quanto mais o desempenho
cai, mais importante é aquela variável para a IA.

In [ ]:
pipeline_nb_estresse = Pipeline([("prep", StandardScaler()), ("nb", GaussianNB())])
pipeline_nb_estresse.fit(X_train, y_train["risco_estresse"])

result = permutation_importance(
    pipeline_nb_estresse, X_test, y_test["risco_estresse"],
    n_repeats=10, random_state=RANDOM_STATE, scoring="f1_macro"
)
importancias = pd.Series(result.importances_mean, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
importancias.plot(kind="barh", ax=ax)
ax.set_title("Importância das variáveis (Naive Bayes) — risco_estresse")
ax.set_xlabel("Queda no F1 ao embaralhar a coluna")
plt.tight_layout()
plt.show()

## 14. Salvar e reutilizar o modelo

Salva tudo que é preciso para usar o modelo depois (sem re-treinar):
o pipeline treinado, os `encoders` (para traduzir 0/1/2 de volta para
"Baixo"/"Médio"/"Alto") e a lista de features na ordem certa.

In [ ]:
joblib.dump({
    "pipeline": pipeline_nb,
    "encoders": encoders,
    "features": FEATURES,
    "label_cols": LABEL_COLS,
}, "modelo_naive_bayes.joblib")

print("Modelo salvo em modelo_naive_bayes.joblib")

# Para carregar depois (em outro notebook/script), sem precisar re-treinar:
# dados = joblib.load("modelo_naive_bayes.joblib")
# pipeline = dados["pipeline"]
# pipeline.predict(nova_linha_de_dados)

## Resumo — como saber se o modelo está bom, na prática

Um checklist rápido para usar sempre que mexer no modelo ou nos dados:

1. **F1-score por classe** (seção 7) acima de ~0,70 em todas as classes,
   sem nenhuma muito abaixo das outras.
2. **Matriz de confusão** (seção 7) com a maior parte dos acertos na
   diagonal, e sem erros graves (confundir "Baixo" com "Alto").
3. **Curva de aprendizado** (seção 9) com a linha de validação subindo e
   se aproximando da linha de treino conforme mais dados são usados.
4. **Validação cruzada** (seção 10) com desvio padrão baixo entre os
   folds — mostra que o resultado não depende de sorte na divisão dos
   dados.
5. **Testes de sanidade** (seção 12) passando sempre — pior caso dá
   "Alto", melhor caso dá "Baixo".
6. Sempre que adicionar dados novos (ex.: feedback de profissionais) e
   re-treinar, rode as seções 7, 9, 10 e 12 de novo e compare com os
   números desta vez, para garantir que o modelo não piorou.